# 04 — Build Gold Features

Build a modeling-ready 10,100-cell feature table from the preserved combined-grid snapshot. The table replaces legacy filesystem access with governed Delta inputs, maps every source-assigned district to the canonical Silver reference, and keeps source-missing district assignments and analytical nulls visible for later policy decisions.

## 1. Runtime and feature contract

`gold_grid_features` contains one row per legacy 250 m grid cell. It retains land use, building height, public transport, 2023 rent, EMVS housing, and EPSG:3035 WKB. The source GeoPackage is a migration baseline derived by the legacy workflow; this table does not claim to reconstruct its upstream spatial processing.

Feature values are renamed for analytical clarity but are not imputed or scaled. Missing-value counts are reported so the later ML slice can choose and document an explicit preparation policy.

In [ ]:
import importlib
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

cwd = Path.cwd()
repo_root = next(
    (candidate for candidate in [cwd, *cwd.parents] if (candidate / "src" / "urban_workflow").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate the bundle root containing src/urban_workflow.")

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

sys.modules.pop("urban_workflow.gold.table_operations", None)
importlib.invalidate_caches()

from urban_workflow.gold.table_operations import (
    add_gold_metadata,
    build_district_features,
    build_grid_features,
    grid_feature_null_profile,
    validate_district_features,
    validate_grid_features,
    validate_persisted_district_feature_table,
    validate_persisted_gold_table,
    write_snapshot_table,
)

print(f"Bundle root: {repo_root}")

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "urban_workflow")
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
if not all((CATALOG, SCHEMA)):
    raise ValueError("catalog and schema must be non-empty.")
BRONZE_GRID_TABLE = f"{CATALOG}.{SCHEMA}.bronze_combined_grid_snapshot"
SILVER_DISTRICT_TABLE = f"{CATALOG}.{SCHEMA}.silver_district_reference"
SILVER_BOUNDARY_TABLE = f"{CATALOG}.{SCHEMA}.silver_district_boundaries"
SILVER_POPULATION_TABLE = f"{CATALOG}.{SCHEMA}.silver_population"
SILVER_EMVS_TABLE = f"{CATALOG}.{SCHEMA}.silver_emvs_housing"
SILVER_INDICATOR_TABLE = f"{CATALOG}.{SCHEMA}.silver_indicator_panel"
GOLD_GRID_TABLE = f"{CATALOG}.{SCHEMA}.gold_grid_features"
GOLD_DISTRICT_TABLE = f"{CATALOG}.{SCHEMA}.gold_district_features"

gold_started_at = datetime.now(timezone.utc)
gold_run_id = (
    f"gold_{gold_started_at.strftime('%Y%m%dT%H%M%SZ')}_"
    f"{uuid4().hex[:8]}"
)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Gold run: {gold_run_id}")
print(f"Grid source: {BRONZE_GRID_TABLE}")
print(f"District reference: {SILVER_DISTRICT_TABLE}")
print(f"Grid target: {GOLD_GRID_TABLE}")
print(f"District target: {GOLD_DISTRICT_TABLE}")

## 2. Canonical grid features

The source district label is normalized with the same accent, punctuation, and spacing rule used in Silver, then joined to the authoritative 21-row district reference. The original source label and normalized key remain available beside the canonical code and name. The legacy snapshot contains 433 cells without a source district label; they remain present with null canonical district fields and `district_mapping_status=source_district_missing`. All 9,667 non-null source labels must resolve as `canonical_match`.

In [ ]:
bronze_grid = spark.table(BRONZE_GRID_TABLE)
district_reference = spark.table(SILVER_DISTRICT_TABLE)
gold_grid = build_grid_features(bronze_grid, district_reference)
grid_validation = validate_grid_features(bronze_grid, gold_grid)
null_profile = grid_feature_null_profile(gold_grid)
gold_grid = add_gold_metadata(
    gold_grid,
    gold_run_id=gold_run_id,
    processed_at_utc=gold_started_at,
)

print(grid_validation)
print({"feature_null_counts": null_profile})
display(
    gold_grid.select(
        "cell_id", "district_code", "district_name",
        "district_mapping_status",
        "land_use_class_simplified", "building_height_mean_m",
        "public_transport_stop_count",
        "has_good_public_transport_access",
        "rent_median_eur_m2_2023", "emvs_housing_units_total",
    ).orderBy("cell_id").limit(100)
)

## 3. Canonical district features

The 21-row district table combines valid Silver boundary area, 2024 district population, EMVS housing, eight reviewed district indicators, and aggregates from the 9,667 canonically mapped grid cells. Each indicator uses the newest observation year, newest panel snapshot, observation date, and source record as a deterministic ordering rule; a selected source conflict fails validation. Height and rent averages use observed values only, with contributing-cell counts and coverage shares beside them. The 433 source-unassigned grid cells remain in the grid table and do not enter district metrics.

In [ ]:
district_boundaries = spark.table(SILVER_BOUNDARY_TABLE)
silver_population = spark.table(SILVER_POPULATION_TABLE)
silver_emvs = spark.table(SILVER_EMVS_TABLE)
silver_indicators = spark.table(SILVER_INDICATOR_TABLE)
district_features = build_district_features(
    district_reference,
    district_boundaries,
    silver_population,
    silver_emvs,
    silver_indicators,
    gold_grid,
)
district_validation = validate_district_features(gold_grid, district_features)
district_features = add_gold_metadata(
    district_features,
    gold_run_id=gold_run_id,
    processed_at_utc=gold_started_at,
)

print(district_validation)
display(
    district_features.select(
        "district_code", "district_name", "area_km2",
        "population_total", "population_density_per_km2",
        "housing_units_total", "housing_units_per_1000_residents",
        "grid_cell_count", "grid_height_coverage_share",
        "grid_building_height_mean_m_avg",
        "grid_good_public_transport_access_share",
        "grid_green_like_share", "grid_rent_coverage_share",
        "grid_rent_median_eur_m2_2023",
        "green_area_per_10000", "income_per_person",
        "unemployment_rate", "vulnerability_index",
        "district_indicator_coverage_share",
        "district_feature_coverage_score",
    ).orderBy("district_code")
)

## 4. Persist and read back

Both fixed Gold snapshots are overwritten deterministically. The read-back gate repeats the grid contract and the 21-district identity, source coverage, aggregation, feature-range, geometry, and multi-source lineage checks against the persisted Delta tables.

In [ ]:
write_snapshot_table(gold_grid, GOLD_GRID_TABLE)
write_snapshot_table(district_features, GOLD_DISTRICT_TABLE)
spark.sql(
    f"COMMENT ON TABLE {GOLD_GRID_TABLE} IS "
    "'Modeling-ready Madrid grid features migrated from the governed legacy derived snapshot.'"
)
spark.sql(
    f"COMMENT ON TABLE {GOLD_DISTRICT_TABLE} IS "
    "'Canonical Madrid district features from Silver entities and mapped Gold grid cells.'"
)

persisted_grid = validate_persisted_gold_table(
    spark,
    table_name=GOLD_GRID_TABLE,
    expected_record_count=10_100,
)
persisted_validation = validate_grid_features(bronze_grid, persisted_grid)
persisted_null_profile = grid_feature_null_profile(persisted_grid)
persisted_districts = validate_persisted_district_feature_table(
    spark,
    table_name=GOLD_DISTRICT_TABLE,
)
persisted_district_validation = validate_district_features(
    persisted_grid,
    persisted_districts,
)

print(persisted_validation)
print({"persisted_feature_null_counts": persisted_null_profile})
print(persisted_district_validation)
print(
    f"Validated {persisted_grid.count()} Gold grid feature records and "
    f"{persisted_districts.count()} Gold district feature records."
)
dbutils.notebook.exit(
    "SUCCESS: Validated 10,100 grid and 21 district feature records."
)

## 5. Slice status

The grid slice has passed live acceptance. The completed district contract is accepted when all 21 districts pass the live write and read-back gate with 9,667 mapped grid cells, deterministic conflict-free indicator selections, valid coverage shares, and complete multi-source lineage.